# Classificação de Dígitos — SVHN

Grupo: Samuel, Wilker e Bernardo.

**Como rodar no Colab:** `Ambiente de execução → Alterar o tipo de ambiente → GPU`, depois `Executar tudo`.

**Protocolo**
- `split="train"` é dividido em 90% treino e 10% validação, com semente fixa (`SEED = 42`).
- A validação acompanha os experimentos: cada experimento é avaliado na época com melhor acurácia de validação.
- `split="test"` só informa a acurácia; não participa de nenhuma escolha.
- Cada experimento muda **uma coisa** em relação a um anterior, para isolar o efeito.

In [ ]:
import json
import math
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets

SEED = 42
EPOCHS = 15
DATA_DIR = Path("data")
RESULTS_DIR = Path("results")
WEIGHTS_PATH = Path("weights/best.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

# Opcional: salvar os resultados no Google Drive para não perder nada se o Colab desconectar.
# Experimentos já salvos em RESULTS_DIR são carregados em vez de retreinados.
# from google.colab import drive; drive.mount("/content/drive")
# RESULTS_DIR = Path("/content/drive/MyDrive/svhn/results")
# WEIGHTS_PATH = Path("/content/drive/MyDrive/svhn/weights/best.pt")

# Modo de teste do CI (SVHN_SMOKE=1): dados aleatórios, poucas épocas e saídas em smoke/, sem baixar o SVHN.
# Sem a variável, nada muda.
SMOKE = os.environ.get("SVHN_SMOKE") == "1"
if SMOKE:
    EPOCHS = 2
    RESULTS_DIR, WEIGHTS_PATH = Path("smoke/results"), Path("smoke/weights/best.pt")

# Mesma semente -> mesmos resultados também na GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| modo de teste (dados aleatórios)" if SMOKE else "")

## Dados

O dataset inteiro cabe na memória da GPU (~300 MB em `uint8`), então os batches são montados direto
na GPU, sem `DataLoader`. A média e o desvio da normalização são calculados **só no conjunto de treino**.

In [ ]:
def load_split(split):
    if SMOKE:  # mesmo formato do SVHN: uint8 N x 3 x 32 x 32 e rótulos de 0 a 9
        g = torch.Generator().manual_seed(len(split))
        n = 3000 if split == "train" else 1000
        return (torch.randint(0, 256, (n, 3, 32, 32), dtype=torch.uint8, generator=g),
                torch.randint(0, 10, (n,), generator=g))
    ds = datasets.SVHN(DATA_DIR, split=split, download=True)
    return torch.from_numpy(ds.data), torch.from_numpy(ds.labels).long()  # data: N x 3 x 32 x 32, uint8


x_train_full, y_train_full = load_split("train")
x_test, y_test = load_split("test")

perm = torch.randperm(len(x_train_full), generator=torch.Generator().manual_seed(SEED))
n_val = len(perm) // 10
x_val, y_val = x_train_full[perm[:n_val]], y_train_full[perm[:n_val]]
x_train, y_train = x_train_full[perm[n_val:]], y_train_full[perm[n_val:]]

STD, MEAN = (t.view(1, 3, 1, 1).to(DEVICE) for t in torch.std_mean(x_train.float().div(255), dim=(0, 2, 3)))

x_train, y_train, x_val, y_val, x_test, y_test = (
    t.to(DEVICE) for t in (x_train, y_train, x_val, y_val, x_test, y_test))
print(f"treino {len(x_train)} | validação {len(x_val)} | teste {len(x_test)}")


def normalize(x):
    return (x.float() / 255 - MEAN) / STD


def augment(x):
    """Rotação de ±8°, escala de ±10% e translação de ±8% aleatórias por imagem, feitas na GPU."""
    n, dev = len(x), x.device
    rand = lambda: torch.rand(n, device=dev) * 2 - 1
    angle, scale = rand() * math.radians(8), 1 + rand() * 0.1
    tx, ty = rand() * 0.16, rand() * 0.16  # o grid vai de -1 a 1, então 8% da largura = 0.16
    cos, sin = torch.cos(angle) / scale, torch.sin(angle) / scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)  # 0 = cor média após normalizar

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(x_train[i].permute(1, 2, 0).cpu())
    axes[0, i].set_title(int(y_train[i]))
    axes[1, i].imshow((augment(normalize(x_train[i:i + 1])) * STD + MEAN)[0].clamp(0, 1).permute(1, 2, 0).cpu())
for ax in axes.flat:
    ax.axis("off")
axes[1, 0].set_title("augment", loc="left")
plt.show()

## Modelos

In [ ]:
def linear():
    return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 10))


def mlp(hidden=(512, 256), dropout=0.0, batchnorm=False):
    layers, size = [nn.Flatten()], 3 * 32 * 32
    for h in hidden:
        layers.append(nn.Linear(size, h))
        if batchnorm:
            layers.append(nn.BatchNorm1d(h))
        layers += [nn.ReLU(), nn.Dropout(dropout)]
        size = h
    return nn.Sequential(*layers, nn.Linear(size, 10))


def cnn(widths=(32, 64, 128), dropout=0.3):
    """3 blocos [conv-BN-ReLU x2 + maxpool]: 32x32 -> 4x4, seguidos de uma camada densa."""
    layers, cin = [], 3
    for w in widths:
        layers += [nn.Conv2d(cin, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(),
                   nn.Conv2d(w, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(), nn.MaxPool2d(2)]
        cin = w
    return nn.Sequential(*layers, nn.Flatten(), nn.Dropout(dropout), nn.Linear(cin * 4 * 4, 256), nn.ReLU(),
                         nn.Dropout(dropout), nn.Linear(256, 10))

## Treino

AdamW (com `weight_decay=0` equivale ao Adam) e LR com decaimento cosseno ao longo das épocas.
A loss de validação é sempre a entropia cruzada pura, mesmo com label smoothing, para comparar experimentos.

In [ ]:
def batches(x, y, size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), size):
        j = idx[i:i + size]
        yield x[j], y[j]


def run_epoch(model, x, y, batch=1024, opt=None, aug=False, label_smoothing=0.0):
    """Uma passada sobre (x, y). Treina se `opt` for dado; senão só avalia. Retorna (loss, acurácia)."""
    model.train(opt is not None)
    total_loss, correct = torch.zeros((), device=x.device), torch.zeros((), device=x.device)
    with torch.set_grad_enabled(opt is not None):
        for xb, yb in batches(x, y, batch, shuffle=opt is not None):
            xb = normalize(xb)
            if aug:
                xb = augment(xb)
            out = model(xb)
            loss = F.cross_entropy(out, yb, label_smoothing=label_smoothing)
            if opt:
                opt.zero_grad()
                loss.backward()
                opt.step()
            total_loss += loss.detach() * len(yb)
            correct += (out.argmax(1) == yb).sum()
    return total_loss.item() / len(y), correct.item() / len(y)


ARCHS = {"linear": linear, "mlp": mlp, "cnn": cnn}


def full_config(cfg):
    """Completa `cfg` com os valores padrão. A config completa é a chave do cache de resultados."""
    cfg = {"arch_args": {}, "batch": 64, "epochs": EPOCHS, "augment": False, "weight_decay": 0.0,
           "label_smoothing": 0.0, "seed": SEED, **cfg}
    if SMOKE:  # cnn_30ep continua com mais épocas que as outras, mas sem as 30
        cfg["epochs"] = min(cfg["epochs"], EPOCHS + 1)
    return cfg


def setup(cfg):
    """Modelo, otimizador e LR cosseno ao longo de cfg["epochs"], com a semente da config."""
    torch.manual_seed(cfg["seed"])
    model = ARCHS[cfg["arch"]](**cfg["arch_args"]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    return model, opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg["epochs"])


def train(cfg):
    """Treina conforme `cfg` (já completa) e devolve (modelo na melhor época de validação, histórico, resumo)."""
    epochs = cfg["epochs"]
    model, opt, sched = setup(cfg)
    hist = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc")}
    best_val, best_epoch, best_state = -1.0, 0, None
    start = time.time()
    for ep in range(1, epochs + 1):
        run_epoch(model, x_train, y_train, cfg["batch"], opt, cfg["augment"], cfg["label_smoothing"])
        sched.step()
        # Métricas de treino medidas em modo de avaliação (sem dropout/augment), comparáveis às de validação.
        tl, ta = run_epoch(model, x_train, y_train)
        vl, va = run_epoch(model, x_val, y_val)
        for k, v in zip(hist, (tl, ta, vl, va)):
            hist[k].append(v)
        if va > best_val:
            best_val, best_epoch = va, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"  ep {ep:2d}/{epochs} | treino loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}")
    model.load_state_dict(best_state)
    _, test_acc = run_epoch(model, x_test, y_test)
    b = best_epoch - 1
    summary = dict(epocas=epochs, melhor_epoca=best_epoch,
                   train_loss=hist["train_loss"][b], val_loss=hist["val_loss"][b],
                   train_acc=hist["train_acc"][b], val_acc=best_val, test_acc=test_acc,
                   tempo_s=round(time.time() - start))
    return model, hist, summary


def train_final(cfg):
    """Treina em todo o split="train" (treino + validação) até cfg["stop_epoch"].

    O LR segue o mesmo cosseno de cfg["epochs"] épocas do experimento, interrompido na época da melhor
    validação, para repetir o caminho de otimização que a validação escolheu. Fica o modelo da última época.
    """
    x_tr, y_tr = torch.cat([x_train, x_val]), torch.cat([y_train, y_val])
    model, opt, sched = setup(cfg)
    start = time.time()
    for ep in range(1, cfg["stop_epoch"] + 1):
        run_epoch(model, x_tr, y_tr, cfg["batch"], opt, cfg["augment"], cfg["label_smoothing"])
        sched.step()
        print(f"  ep {ep:2d}/{cfg['stop_epoch']}")
    train_loss, train_acc = run_epoch(model, x_tr, y_tr)
    _, test_acc = run_epoch(model, x_test, y_test)
    return model, dict(train_loss=train_loss, train_acc=train_acc, test_acc=test_acc,
                       tempo_s=round(time.time() - start))

## Experimentos

| # | Experimento | Mudança (em relação a) |
|---|---|---|
| 1 | linear | classificador linear 3072 → 10 |
| 2 | mlp_baseline | MLP 512-256, ReLU, Adam lr=1e-3, batch 64 |
| 3 | mlp_lr_1e-4 | lr 1e-4 (2) |
| 4 | mlp_lr_1e-2 | lr 1e-2 (2) |
| 5 | mlp_batch_256 | batch 256 (2) |
| 6 | mlp_dropout | + Dropout 0.3 (2) |
| 7 | mlp_bn_dropout | + BatchNorm (6) |
| 8 | mlp_deep | camadas 1024-512-256 (7) |
| 9 | cnn | CNN 3 blocos 32/64/128 filtros |
| 10 | cnn_augment | + data augmentation (9) |
| 11 | cnn_wide | filtros 64/128/256 (10) |
| 12 | cnn_wd | weight decay 0.05 (10) |
| 13 | cnn_30ep | 30 épocas (10) |
| 14 | cnn_ls | label smoothing 0.1 (10) |
| 15 | cnn_best | (10) + as mudanças de 11–14 que melhoraram a validação |

In [ ]:
# Configs só com dados (listas, não tuplas), para poderem ser comparadas com as salvas em JSON.
MLP = dict(arch="mlp", lr=1e-3)
CNN_AUG = dict(arch="cnn", lr=1e-3, augment=True)
CNN_VARIATIONS = {
    "cnn_wide": dict(arch_args={"widths": [64, 128, 256]}),
    "cnn_wd":   dict(weight_decay=0.05),
    "cnn_30ep": dict(epochs=30),
    "cnn_ls":   dict(label_smoothing=0.1),
}
EXPERIMENTS = {
    "linear":         dict(arch="linear", lr=1e-3),
    "mlp_baseline":   MLP,
    "mlp_lr_1e-4":    {**MLP, "lr": 1e-4},
    "mlp_lr_1e-2":    {**MLP, "lr": 1e-2},
    "mlp_batch_256":  {**MLP, "batch": 256},
    "mlp_dropout":    {**MLP, "arch_args": {"dropout": 0.3}},
    "mlp_bn_dropout": {**MLP, "arch_args": {"dropout": 0.3, "batchnorm": True}},
    "mlp_deep":       {**MLP, "arch_args": {"hidden": [1024, 512, 256], "dropout": 0.3, "batchnorm": True}},
    "cnn":            dict(arch="cnn", lr=1e-3),
    "cnn_augment":    CNN_AUG,
    **{name: {**CNN_AUG, **change} for name, change in CNN_VARIATIONS.items()},
}
results, histories, configs = {}, {}, {}


def run(name, cfg):
    """Treina e salva em RESULTS_DIR; se já foi salvo com exatamente a mesma config completa, só carrega."""
    cfg = full_config(cfg)
    path = RESULTS_DIR / f"{name}.json"
    saved = json.loads(path.read_text()) if path.exists() else {}
    if saved.get("config") == cfg:
        print(f"{name}: carregado de {path}")
    else:
        print(f"{name}:")
        _, hist, summary = train(cfg)
        saved = dict(summary=summary, hist=hist, config=cfg)
        path.write_text(json.dumps(saved))
    results[name], histories[name], configs[name] = saved["summary"], saved["hist"], cfg
    s = saved["summary"]
    print(f"  -> val {s['val_acc']:.4f} (época {s['melhor_epoca']}) | teste {s['test_acc']:.4f}")

In [ ]:
for name, cfg in EXPERIMENTS.items():
    run(name, cfg)

### Experimento 15 — combinação
Junta à `cnn_augment` só as mudanças de 11–14 que superaram a `cnn_augment` na **validação**.

In [ ]:
improved = [n for n in CNN_VARIATIONS if results[n]["val_acc"] > results["cnn_augment"]["val_acc"]]
print("mudanças que melhoraram a validação:", improved or "nenhuma")
cnn_best = dict(CNN_AUG)
for n in improved:
    cnn_best.update(CNN_VARIATIONS[n])
if improved:
    run("cnn_best", cnn_best)
else:  # sem mudanças, cnn_best == cnn_augment: reaproveita em vez de retreinar
    for d in (results, histories, configs):
        d["cnn_best"] = d["cnn_augment"]

## Resultados

In [ ]:
table = pd.DataFrame.from_dict(results, orient="index").rename_axis("modificacao").reset_index()
table.insert(0, "experimento", range(1, len(table) + 1))
table.to_csv(RESULTS_DIR / "results.csv", index=False)
table.style.format(precision=4).highlight_max(subset=["val_acc"], color="#c6efce")

In [ ]:
def plot_curves(name):
    h = histories[name]
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 3.5))
    for ax, key, title in ((ax_loss, "loss", "Loss"), (ax_acc, "acc", "Acurácia")):
        epochs = range(1, len(h[f"train_{key}"]) + 1)
        ax.plot(epochs, h[f"train_{key}"], label="treino")
        ax.plot(epochs, h[f"val_{key}"], label="validação")
        ax.axvline(results[name]["melhor_epoca"], color="gray", ls=":", label="melhor época")
        ax.set(title=f"{name} — {title}", xlabel="época")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"{name}.png", dpi=120)
    plt.show()


for name in results:
    plot_curves(name)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for name in ("linear", "mlp_baseline", "mlp_bn_dropout", "cnn", "cnn_augment", "cnn_best"):
    ax.plot(range(1, len(histories[name]["val_acc"]) + 1), histories[name]["val_acc"], label=name)
ax.set(title="Acurácia de validação por época", xlabel="época", ylabel="acurácia")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "comparacao_validacao.png", dpi=120)
plt.show()

best = table.loc[table["val_acc"].idxmax()]
print(f"Melhor na validação: {best['modificacao']} | val {best['val_acc']:.4f} | teste {best['test_acc']:.4f}")

## Modelo final

O modelo final é o experimento com a **maior acurácia de validação**. Ele é retreinado com todo o
`split="train"` (treino + validação) com a mesma config e semente, parando na época em que atingiu a
melhor validação (com o mesmo agendamento de LR). Como não sobra validação, fica o modelo da última época.
O teste só informa a acurácia.

Os pesos são salvos em `WEIGHTS_PATH` junto com a config, a média e o desvio da normalização, que a
predição do conjunto adicional precisa. Se o arquivo já existe com a mesma config, não retreina.

In [ ]:
FINAL = best["modificacao"]  # o melhor na validação, escolhido na tabela de resultados
final_cfg = {**configs[FINAL], "stop_epoch": results[FINAL]["melhor_epoca"]}
print(f"Modelo final: {FINAL} | val {results[FINAL]['val_acc']:.4f} | "
      f"{final_cfg['stop_epoch']} de {final_cfg['epochs']} épocas")
print(json.dumps(final_cfg, indent=2))

try:
    ckpt = torch.load(WEIGHTS_PATH, map_location="cpu")
except Exception:  # ausente ou corrompido (ex.: Colab caiu ao salvar)
    ckpt = {}
if ckpt.get("config") == final_cfg:
    print(f"carregado de {WEIGHTS_PATH}")
else:
    trained, final_summary = train_final(final_cfg)
    ckpt = dict(experimento=FINAL, config=final_cfg, mean=MEAN.cpu(), std=STD.cpu(),
                state_dict={k: v.cpu() for k, v in trained.state_dict().items()},
                summary=final_summary, val_acc_experimento=results[FINAL]["val_acc"])
    WEIGHTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = WEIGHTS_PATH.with_suffix(".tmp")
    torch.save(ckpt, tmp)
    tmp.replace(WEIGHTS_PATH)  # troca atômica: nunca fica um best.pt pela metade
    print(f"pesos salvos em {WEIGHTS_PATH}")

In [ ]:
# Confere que os pesos salvos reproduzem a acurácia de teste (é o mesmo arquivo usado na predição).
# Tolerância de 0,1 ponto: em outro dispositivo (GPU do Colab vs Mac/CPU) algumas predições podem mudar.
final_model = ARCHS[ckpt["config"]["arch"]](**ckpt["config"]["arch_args"]).to(DEVICE)
final_model.load_state_dict(ckpt["state_dict"])
_, test_acc = run_epoch(final_model, x_test, y_test)
assert abs(test_acc - ckpt["summary"]["test_acc"]) < 1e-3, (test_acc, ckpt["summary"]["test_acc"])

final_info = dict(experimento=ckpt["experimento"], arquitetura=str(final_model), config=ckpt["config"],
                  parametros=sum(p.numel() for p in final_model.parameters()),
                  val_acc=ckpt["val_acc_experimento"], test_acc=test_acc,
                  train_acc=ckpt["summary"]["train_acc"], train_loss=ckpt["summary"]["train_loss"])
(RESULTS_DIR / "final.json").write_text(json.dumps(final_info, indent=2))
print(final_model)
print(f"{final_info['parametros']:,} parâmetros | val (experimento) {final_info['val_acc']:.4f} | teste {test_acc:.4f}")